# Joint BiomedClinicalBERT committee + biomed-r1 escalation inference

This notebook:

1. Loads the winning calibrated committee (Pipeline S: Simple supervised)  
2. Runs committee inference on the **500-example test split**  
3. Escalates the least confident cases to **BioMed-R1-8B**  
4. Uses **dev-side expert examples only** for few-shot prompting  
5. Reports baseline committee metrics and final joint-system metrics


In [1]:
import gc
import json
import re
from pathlib import Path
import numpy as np
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.95"

import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from Stage2_common_training_utils import load_json
from Stage2_expert_stage_utils import (
    get_tokenizer,
    load_expert_json,
    add_expert_label,
    predict_calibrated_probs_for_dataset,
    evaluate_probs,
    id_to_label,
    label_to_id,
)

import inspect
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file


In [2]:
# paths

winning_test_result_path = "./stage2_cv_results_final/winning_test_result.json"

official_dev_path = "./stage2_cv_results/fixed_splits/dev_500.json"
official_test_path = "./stage2_cv_results/fixed_splits/test_500.json"

# reasoning model
reasoning_base_model_id = "zou-lab/BioMed-R1-8B"
reasoning_adapter_path = "./final_adapter_biomedr18b"

# inference config
committee_batch_size = 4
reasoning_max_new_tokens = 128
reasoning_prompt_max_length = 2048
max_escalation_fraction = 0.2

escalation_confidence_threshold = 0.55
escalation_margin_threshold = 0.22
# device config
reasoning_device_preference = "mps"

# few-shot prompt design: 1 no, 1 maybe, 1 yes
few_shot_plan = {0: 1, 1: 1, 2: 1}

# save location
joint_results_path = "./stage2_cv_results_final/joint_committee_biomed_r1_results.json"

In [3]:
winning_test_result = load_json(winning_test_result_path)
committee_members = winning_test_result["committee_members"]

tokenizer = get_tokenizer()
dev_500 = load_expert_json(official_dev_path).map(add_expert_label)
test_500 = load_expert_json(official_test_path).map(add_expert_label)

print(f"committee members: {len(committee_members)}")
print(f"dev examples: {len(dev_500)}")
print(f"test examples: {len(test_500)}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

committee members: 10
dev examples: 500
test examples: 500


In [4]:
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def print_mem(tag=""):
    if torch.backends.mps.is_available():
        current_gb = torch.mps.current_allocated_memory() / (1024 ** 3)
        driver_gb = torch.mps.driver_allocated_memory() / (1024 ** 3)
        recommended_gb = torch.mps.recommended_max_memory() / (1024 ** 3)
        print(
            f"[{tag}] MPS current={current_gb:.2f} GB | "
            f"driver={driver_gb:.2f} GB | recommended_max={recommended_gb:.2f} GB"
        )


def load_committee_mean_probs(committee_members, dataset, tokenizer, batch_size=8):
    all_member_probs = []

    for member_idx, member in enumerate(committee_members):
        member_probs = predict_calibrated_probs_for_dataset(
            checkpoint_path=member["best_checkpoint"],
            dataset=dataset,
            tokenizer=tokenizer,
            temperature=member["temperature"],
            batch_size=batch_size,
        )
        all_member_probs.append(member_probs)
        cleanup_memory()
        print(f"loaded committee member {member_idx + 1}/{len(committee_members)}")

    all_member_probs = np.stack(all_member_probs, axis=0)
    mean_probs = all_member_probs.mean(axis=0)
    preds = mean_probs.argmax(axis=1)
    confidence = mean_probs.max(axis=1)
    sorted_probs = np.sort(mean_probs, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    return all_member_probs, mean_probs, preds, confidence, margin


def evaluate_pred_array(preds, labels):
    per_class_f1 = f1_score(
        labels,
        preds,
        average=None,
        labels=[0, 1, 2],
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "f1_no": float(per_class_f1[0]),
        "f1_maybe": float(per_class_f1[1]),
        "f1_yes": float(per_class_f1[2]),
        "confusion_matrix": confusion_matrix(labels, preds, labels=[0, 1, 2]).tolist(),
    }

def load_baseline_from_winning_artifact(
    winning_test_result,
    dataset,
    tokenizer,
    batch_size=8,
    prefer_stored_outputs=True,
):
    """
    Use stored baseline committee outputs from winning_test_result.json when available.
    Fallback: recompute from committee checkpoints if stored outputs are missing.
    """
    labels = np.array(dataset["labels"])

    stored_preds = winning_test_result.get("test_preds")
    stored_mean_probs = winning_test_result.get("mean_probs")

    if prefer_stored_outputs and stored_preds is not None and stored_mean_probs is not None:
        mean_probs = np.array(stored_mean_probs, dtype=np.float32)
        preds = np.array(stored_preds, dtype=np.int64)

        if len(preds) != len(dataset):
            raise ValueError(
                "Stored test_preds length does not match the provided test dataset length."
            )
        if mean_probs.shape[0] != len(dataset):
            raise ValueError(
                "Stored mean_probs length does not match the provided test dataset length."
            )

        confidence = mean_probs.max(axis=1)
        sorted_probs = np.sort(mean_probs, axis=1)
        margin = sorted_probs[:, -1] - sorted_probs[:, -2]
        metrics = evaluate_pred_array(preds, labels)

        print("using stored baseline committee outputs from winning_test_result.json")
        return None, mean_probs, preds, confidence, margin, metrics

    print("stored outputs unavailable -> recomputing baseline from committee checkpoints")
    all_member_probs, mean_probs, preds, confidence, margin = load_committee_mean_probs(
        committee_members=winning_test_result["committee_members"],
        dataset=dataset,
        tokenizer=tokenizer,
        batch_size=batch_size,
    )
    metrics = evaluate_pred_array(preds, labels)
    return all_member_probs, mean_probs, preds, confidence, margin, metrics

def pick_few_shot_examples(dev_dataset, few_shot_plan):
    selected = []

    for class_id, number_needed in few_shot_plan.items():
        class_rows = [row for row in dev_dataset if row["labels"] == class_id]
        class_rows = sorted(class_rows, key=lambda row: len(row["CONTEXTS"]))
        selected.extend(class_rows[:number_needed])

    return selected


def format_few_shot_example(row):
    answer = id_to_label[row["labels"]]
    return (
        f"question: {row['QUESTION']}\n"
        f"context: {row['CONTEXTS']}\n"
        f"answer: {answer}"
    )


def build_reasoning_prompt(question, context, few_shot_examples):
    examples_text = "\n\n".join(format_few_shot_example(row) for row in few_shot_examples)

    prompt = f"""You are a biomedical question answering model.

    Read the question and abstract carefully.
    Return exactly one final answer label from this set only:
    (yes, no, maybe)
    
    Use 'maybe' when the abstract is genuinely inconclusive, mixed, or not strong enough to support a clear yes/no answer.
    Do not explain. Output only one word: yes, no, or maybe.
    
    See some examples:
    {examples_text}
    
    Now, for this next question and context, predict the final answer label.
    
    question: {question}
    context: {context}
    answer:"""

    return prompt


def extract_label_from_text(text):
    text = text.strip().lower()
    text = re.sub(r"<[^>]+>", " ", text)

    matches = re.findall(r"\b(yes|no|maybe)\b", text)
    if matches:
        return matches[-1]

    return None


def resolve_reasoning_device(preference="mps"):
    preference = (preference or "mps").lower()

    if preference == "cpu":
        return torch.device("cpu")
    if preference == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if preference == "mps" and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def pick_reasoning_dtype(device):
    if device.type in {"cuda", "mps"}:
        return torch.float16
    return torch.float32


def load_reasoning_model(base_model_id, adapter_path=None, device_preference="mps"):
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = resolve_reasoning_device(device_preference)
    dtype = pick_reasoning_dtype(device)

    cleanup_memory()
    print_mem("before reasoning model load")

    model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )

    if adapter_path is not None:
        adapter_path = Path(adapter_path)

        chat_template_path = adapter_path / "chat_template.jinja"
        if chat_template_path.exists():
            tokenizer.chat_template = chat_template_path.read_text(encoding="utf-8")

        with open(adapter_path / "adapter_config.json", "r", encoding="utf-8") as f:
            raw_cfg = json.load(f)

        supported = set(inspect.signature(LoraConfig.__init__).parameters)
        cleaned_cfg = {
            k: v for k, v in raw_cfg.items()
            if k in supported and v is not None
        }

        peft_config = LoraConfig(**cleaned_cfg)
        model = get_peft_model(model, peft_config)

        state = load_file(str(adapter_path / "adapter_model.safetensors"))

        fixed_state = {}
        for k, v in state.items():
            if k.startswith("base_model.model.base_model.model."):
                k = k.replace("base_model.model.base_model.model.", "base_model.model.", 1)
            fixed_state[k] = v

        load_result = set_peft_model_state_dict(
            model,
            fixed_state,
            adapter_name="default",
        )

        missing_lora = [k for k in load_result.missing_keys if "lora_" in k]
        unexpected_lora = [k for k in load_result.unexpected_keys if "lora_" in k]

        print("Missing LoRA keys:", len(missing_lora))
        print("Unexpected LoRA keys:", len(unexpected_lora))

        del state, fixed_state, load_result
        cleanup_memory()

        print("Merging LoRA into base model for inference...")
        model = model.merge_and_unload(safe_merge=True)
        cleanup_memory()
        print("LoRA merge complete.")

    model = model.to(device)
    model.eval()

    cleanup_memory()
    print_mem("after reasoning model load")
    return tokenizer, model, device


@torch.inference_mode()
def run_reasoning_model(
    reasoning_model,
    reasoning_tokenizer,
    device,
    prompt,
    max_new_tokens=48,
    prompt_max_length=1024,
):
    if getattr(reasoning_tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt}]
        input_ids = reasoning_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        input_ids = input_ids[:, -prompt_max_length:].to(device)
        encoded = {"input_ids": input_ids}
        input_len = input_ids.shape[1]
    else:
        encoded = reasoning_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=prompt_max_length,
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        input_len = encoded["input_ids"].shape[1]

    generated = reasoning_model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=False,
        cache_implementation="offloaded",
        pad_token_id=reasoning_tokenizer.eos_token_id,
        eos_token_id=reasoning_tokenizer.eos_token_id,
    )

    new_tokens = generated[0][input_len:]
    text = reasoning_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    del encoded, generated, new_tokens
    cleanup_memory()

    return text



In [5]:
def should_escalate(
    mean_probs_row,
    confidence_threshold=0.60,
    margin_threshold=0.10,
):
    confidence = float(np.max(mean_probs_row))
    sorted_probs = np.sort(mean_probs_row)
    margin = float(sorted_probs[-1] - sorted_probs[-2])

    return (confidence < confidence_threshold) or (margin < margin_threshold)


def uncertainty_score(mean_probs_row):
    confidence = float(np.max(mean_probs_row))
    sorted_probs = np.sort(mean_probs_row)
    margin = float(sorted_probs[-1] - sorted_probs[-2])

    # higher = more uncertain
    return (1.0 - confidence) + (1.0 - margin)


def select_escalation_indices(
    mean_probs,
    max_escalation_fraction=0.30,
    confidence_threshold=0.60,
    margin_threshold=0.10,
):
    candidate_rows = []

    for row_id, row_probs in enumerate(mean_probs):
        if should_escalate(
            row_probs,
            confidence_threshold=confidence_threshold,
            margin_threshold=margin_threshold,
        ):
            candidate_rows.append((row_id, uncertainty_score(row_probs)))

    candidate_rows = sorted(candidate_rows, key=lambda x: x[1], reverse=True)

    max_escalations = max(1, int(len(mean_probs) * max_escalation_fraction))
    chosen = candidate_rows[:max_escalations]

    return [row_id for row_id, _ in chosen]

In [6]:
all_member_probs, mean_probs, base_preds, base_confidence, base_margin, base_metrics = load_baseline_from_winning_artifact(
    winning_test_result=winning_test_result,
    dataset=test_500,
    tokenizer=tokenizer,
    batch_size=committee_batch_size,
    prefer_stored_outputs=True,
)

test_labels = np.array(test_500["labels"])

print("baseline committee metrics")
for key, value in base_metrics.items():
    if key != "confusion_matrix":
        print(f"  {key}: {value:.4f}")

# free committee member stack before loading the 8B reasoning model
if all_member_probs is not None:
    del all_member_probs
cleanup_memory()
print_mem("after committee inference cleanup")

using stored baseline committee outputs from winning_test_result.json
baseline committee metrics
  accuracy: 0.7020
  macro_f1: 0.5649
  f1_no: 0.7217
  f1_maybe: 0.1709
  f1_yes: 0.8022
[after committee inference cleanup] MPS current=0.00 GB | driver=0.00 GB | recommended_max=24.96 GB


In [7]:
few_shot_examples = pick_few_shot_examples(dev_500, few_shot_plan)
escalation_indices = select_escalation_indices(
    mean_probs=mean_probs,
    max_escalation_fraction=max_escalation_fraction,
    confidence_threshold=escalation_confidence_threshold,
    margin_threshold=escalation_margin_threshold,
)

print(f"few-shot examples used: {len(few_shot_examples)}")
print(f"escalated examples: {len(escalation_indices)} / {len(test_500)}")
print(f"confidence threshold: {escalation_confidence_threshold}")
print(f"margin threshold: {escalation_margin_threshold}")

few-shot examples used: 3
escalated examples: 100 / 500
confidence threshold: 0.55
margin threshold: 0.22


In [8]:
cleanup_memory()
print_mem("before reasoning stage")

reasoning_tokenizer, reasoning_model, reasoning_device = load_reasoning_model(
    reasoning_base_model_id,
    reasoning_adapter_path,
    device_preference=reasoning_device_preference,
)

combined_preds = base_preds.copy()
reasoning_rows = []

for step_idx, row_id in enumerate(escalation_indices, start=1):
    row = test_500[row_id]

    prompt = build_reasoning_prompt(
        question=row["QUESTION"],
        context=row["CONTEXTS"],
        few_shot_examples=few_shot_examples,
    )

    raw_output = run_reasoning_model(
        reasoning_model=reasoning_model,
        reasoning_tokenizer=reasoning_tokenizer,
        device=reasoning_device,
        prompt=prompt,
        max_new_tokens=reasoning_max_new_tokens,
        prompt_max_length=reasoning_prompt_max_length,
    )

    parsed_label = extract_label_from_text(raw_output)
    used_fallback = parsed_label is None

    if parsed_label is None:
        parsed_label = id_to_label[int(base_preds[row_id])]

    combined_preds[row_id] = label_to_id[parsed_label]

    reasoning_rows.append(
        {
            "row_id": int(row_id),
            "base_pred": id_to_label[int(base_preds[row_id])],
            "base_confidence": float(base_confidence[row_id]),
            "base_margin": float(base_margin[row_id]),
            "reasoning_output": raw_output,
            "used_fallback": used_fallback,
            "final_label": parsed_label,
            "gold_label": id_to_label[int(row["labels"])],
        }
    )

    if step_idx <= 5:
        print(
            f"[example {step_idx}] base={id_to_label[int(base_preds[row_id])]} "
            f"final={parsed_label} fallback={used_fallback} raw={repr(raw_output[:200])}"
        )

    if step_idx % 5 == 0 or step_idx == len(escalation_indices):
        print(f"processed {step_idx}/{len(escalation_indices)} escalated examples")
        cleanup_memory()
        print_mem(f"after reasoning example {step_idx}")

changed_count = int(sum(base_preds[row_id] != combined_preds[row_id] for row_id in escalation_indices))
fallback_count = int(sum(row["used_fallback"] for row in reasoning_rows))

print(f"reasoning pass complete for {len(reasoning_rows)} examples")
print(f"changed predictions among escalated rows: {changed_count}")
print(f"fallback-to-committee count: {fallback_count}")

# release the 8B model before metric/report cells
del reasoning_model
cleanup_memory()
print_mem("after reasoning model cleanup")
   

[before reasoning stage] MPS current=0.00 GB | driver=0.00 GB | recommended_max=24.96 GB
[before reasoning model load] MPS current=0.00 GB | driver=0.00 GB | recommended_max=24.96 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Missing LoRA keys: 0
Unexpected LoRA keys: 0
Merging LoRA into base model for inference...
LoRA merge complete.
[after reasoning model load] MPS current=14.96 GB | driver=15.00 GB | recommended_max=24.96 GB
[example 1] base=maybe final=maybe fallback=False raw='answer: maybe'
[example 2] base=maybe final=maybe fallback=False raw='answer: maybe'
[example 3] base=maybe final=maybe fallback=False raw='answer: maybe'
[example 4] base=yes final=maybe fallback=False raw='answer: maybe'
[example 5] base=maybe final=yes fallback=False raw='answer: yes'
processed 5/100 escalated examples
[after reasoning example 5] MPS current=14.96 GB | driver=15.50 GB | recommended_max=24.96 GB
processed 10/100 escalated examples
[after reasoning example 10] MPS current=14.96 GB | driver=15.50 GB | recommended_max=24.96 GB
processed 15/100 escalated examples
[after reasoning example 15] MPS current=14.96 GB | driver=15.50 GB | recommended_max=24.96 GB
processed 20/100 escalated examples
[after reasoning examp

In [9]:
joint_metrics = evaluate_pred_array(combined_preds, test_labels)

print("joint system metrics")
for key, value in joint_metrics.items():
    if key != "confusion_matrix":
        print(f"  {key}: {value:.4f}")

print()
print("metric deltas vs baseline committee")
for key in ["accuracy", "macro_f1", "f1_no", "f1_maybe", "f1_yes"]:
    delta = joint_metrics[key] - base_metrics[key]
    print(f"  {key}: {delta:+.4f}")

joint system metrics
  accuracy: 0.7100
  macro_f1: 0.5638
  f1_no: 0.7175
  f1_maybe: 0.1607
  f1_yes: 0.8133

metric deltas vs baseline committee
  accuracy: +0.0080
  macro_f1: -0.0011
  f1_no: -0.0043
  f1_maybe: -0.0102
  f1_yes: +0.0111


In [10]:
joint_results = {
    "reasoning_base_model_id": reasoning_base_model_id,
    "reasoning_adapter_path": reasoning_adapter_path,
    "winning_test_result_path": winning_test_result_path,
    "official_dev_path": official_dev_path,
    "official_test_path": official_test_path,
    "few_shot_plan": few_shot_plan,
    "max_escalation_fraction": max_escalation_fraction,
    "escalated_count": int(len(escalation_indices)),
    "baseline_metrics": base_metrics,
    "joint_metrics": joint_metrics,
    "reasoning_rows": reasoning_rows,
}

output_path = Path(joint_results_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(joint_results, f, indent=2)

print(f"saved results to: {output_path}")

saved results to: stage2_cv_results_final/joint_committee_biomed_r1_results.json
